In [34]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
STAGING = PROJECT_ROOT / "data" / "staging"

for f in sorted(RAW.iterdir()):
    print(f.name)

.gitkeep
hf_memberships.parquet
hf_persons.parquet
openhalalan_winners.csv
psa_poverty.pdf
roster_legislators.pdf


## 1. hf_persons

In [2]:
persons = pd.read_parquet(RAW / "hf_persons.parquet")
print(persons.shape)
persons.head()
persons.info()

(45424, 4)
<class 'pandas.DataFrame'>
RangeIndex: 45424 entries, 0 to 45423
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           45424 non-null  str  
 1   first_name   45424 non-null  str  
 2   last_name    45424 non-null  str  
 3   name_suffix  1794 non-null   str  
dtypes: str(4)
memory usage: 3.1 MB


In [3]:
print("id unique:", persons["id"].is_unique)
print("duplicate names:", persons.duplicated(subset=["first_name", "last_name", "name_suffix"]).sum())
persons["name_suffix"].value_counts()
persons["last_name"].value_counts().head(15)

id unique: True
duplicate names: 0


last_name
Tan           165
Reyes         165
Garcia        156
Lim           140
Uy            117
De La Cruz    116
Ramos         115
Mendoza       111
Fernandez     109
Villanueva    105
Bautista      101
Flores        100
Dela Cruz      95
Gonzales       93
Santos         88
Name: count, dtype: int64

In [4]:
cruz = persons[persons["last_name"].str.contains("cruz", case=False)]
cruz["last_name"].value_counts()

last_name
De La Cruz       116
Dela Cruz         95
Cruz              85
Vera Cruz          3
Cruzado            3
Cruzat             2
Cruz-Am            2
Sta Cruz           1
Salva Cruz         1
Veracruz           1
Inza Cruz          1
Cruzada            1
Cruza              1
Dela Cruz Jr       1
Escanlar-Cruz      1
Name: count, dtype: int64

In [5]:
print("extra spaces:", (persons["last_name"] != persons["last_name"].str.strip()).sum())
print("all caps:", persons["last_name"].str.isupper().sum())

extra spaces: 0
all caps: 0


In [10]:
suffix_in_last = persons["last_name"].str.contains(r"\s(?:Jr|Sr|II|III|IV)\.?$", regex=True)
print("suffix inside last_name:", suffix_in_last.sum())
persons.loc[suffix_in_last, ["last_name", "name_suffix"]].head(10)


suffix inside last_name: 6


,last_name,name_suffix
9860,Gabuya Jr,NaN
22456,Pineda Jr,NaN
32000,Dela Cruz Jr,NaN
32062,De Manuel Sr,NaN
35868,Lopez Jr,NaN
43472,Rapanut Sr,NaN


In [11]:
persons["name_suffix"].value_counts()

name_suffix
Jr.       1287
Sr.        224
III        135
II          80
V           25
IV          21
I           14
VI           3
VII          1
Jr. IV       1
XVI          1
XIV          1
Jr. II       1
Name: count, dtype: int64

**What it is:** One row per politician. 45,424 rows, 4 columns (`id`, `first_name`, `last_name`, `name_suffix`).

**Quality**
- `id` is unique, so it works as the primary key.
- `first_name` and `last_name` have no missing values, no leading/trailing spaces, and no all-caps entries.
- `name_suffix` is filled for 1,794 rows (about 4%). This is expected, since most people have no suffix.

**Issues found**
- **Surname variants:** the same name is written differently, e.g. "De La Cruz" (116) vs "Dela Cruz" (95), "Vera Cruz" vs "Veracruz", and abbreviations such as "Sta Cruz".
- **Hyphenated surnames** (e.g. "Escanlar-Cruz") may combine two family names.
- **Suffixes inside `last_name`:** 6 rows (e.g. "Lopez Jr", "Rapanut Sr") have the suffix in the last name while `name_suffix` is empty.
- **Odd suffix values:** formatting is consistent ("Jr.", "Sr.", Roman numerals), but "XVI" and "XIV" (1 each) are implausible and "I" (14) is unusual.
- **Zero duplicate full names** across 45,424 people is unusually clean. The source may have merged different people who share a name (to verify using memberships).
- **Common surnames** (Tan 165, Reyes 165, Garcia 156, Lim 140, Uy 117) are shared by unrelated families, so a surname alone is a weak kinship signal.

**Not in this table:** no middle name and no location. Location must come from `hf_memberships`.

## 2. hf_memberships

In [41]:
memberships = pd.read_parquet(RAW / "hf_memberships.parquet")
print(memberships.shape)
memberships.info()

(86234, 8)
<class 'pandas.DataFrame'>
RangeIndex: 86234 entries, 0 to 86233
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         86234 non-null  str  
 1   person_id  86234 non-null  str  
 2   party      86234 non-null  str  
 3   region     86234 non-null  str  
 4   province   86234 non-null  str  
 5   locality   58594 non-null  str  
 6   position   86234 non-null  str  
 7   year       86234 non-null  int32
dtypes: int32(1), str(7)
memory usage: 13.0 MB


In [42]:
orphans = ~memberships["person_id"].isin(persons["id"])
print("memberships with no matching person:", orphans.sum())

memberships with no matching person: 0


In [43]:
memberships["year"].value_counts().sort_index()

year
2004    17372
2007    16610
2010    17507
2013    17164
2016    17581
Name: count, dtype: int64

In [44]:
memberships.loc[memberships["locality"].isna(), "position"].value_counts()

position
COUNCILOR                           17678
PROVINCIAL BOARD MEMBER              3745
MAYOR                                2174
VICE MAYOR                           2170
MEMBER, HOUSE OF REPRESENTATIVES     1082
GOVERNOR                              397
VICE GOVERNOR                         394
Name: count, dtype: int64

In [45]:
pd.crosstab(memberships["position"], memberships["locality"].isna())

locality,False,True
position,,
COUNCILOR,47114,17678
GOVERNOR,0,397
MAYOR,5727,2174
"MEMBER, HOUSE OF REPRESENTATIVES",39,1082
PROVINCIAL BOARD MEMBER,2,3745
VICE GOVERNOR,0,394
VICE MAYOR,5712,2170


In [46]:
town_level = memberships[memberships["position"].isin(["MAYOR", "VICE MAYOR", "COUNCILOR"])]
pd.crosstab(town_level["year"], town_level["locality"].isna())

locality,False,True
year,,
2004,16259,0
2007,15483,0
2010,16380,0
2013,201,15842
2016,10230,6180


In [47]:
town_level.loc[town_level["locality"].isna(), "province"].value_counts().head(10)

province
CEBU             1084
BOHOL             931
CAMARINES SUR     735
BATANGAS          679
CAGAYAN           588
ABRA              540
BULACAN           495
PANGASINAN        478
CAVITE            470
ILOILO            446
Name: count, dtype: int64

In [48]:
t2016 = town_level[town_level["year"] == 2016]
share_missing = t2016["locality"].isna().groupby(t2016["province"]).mean()
share_missing[share_missing > 0].sort_index()

province
ABRA                                   1.000000
AGUSAN DEL NORTE                       1.000000
AGUSAN DEL SUR                         1.000000
AKLAN                                  1.000000
ALBAY                                  1.000000
ANTIQUE                                1.000000
APAYAO                                 1.000000
AURORA                                 1.000000
BASILAN                                1.000000
BATAAN                                 1.000000
BATANES                                1.000000
BATANGAS                               1.000000
BENGUET                                1.000000
BILIRAN                                1.000000
BOHOL                                  1.000000
BUKIDNON                               1.000000
BULACAN                                1.000000
CAGAYAN                                1.000000
CAMARINES NORTE                        1.000000
CAMARINES SUR                          1.000000
CAMIGUIN                       

In [49]:
all_2016 = sorted(t2016["province"].unique())
cutoff = full.index[-1]
before_cutoff = [p for p in all_2016 if p <= cutoff]

print(len(before_cutoff), "provinces from A up to the cutoff")
[p for p in before_cutoff if p not in full.index]

52 provinces from A up to the cutoff


['ILOCOS NORTE',
 'ILOCOS SUR',
 'ILOILO',
 'ISABELA',
 'KALINGA',
 'LA UNION',
 'LAGUNA',
 'LANAO DEL NORTE',
 'LANAO DEL SUR',
 'LEYTE',
 'MAGUINDANAO',
 'MARINDUQUE',
 'MASBATE',
 'MISAMIS OCCIDENTAL',
 'MISAMIS ORIENTAL',
 'MOUNTAIN PROVINCE']

In [50]:
full = share_missing[share_missing == 1].sort_index()
print(len(full), "provinces fully missing, from", full.index[0], "to", full.index[-1])
share_missing[(share_missing > 0) & (share_missing < 1)]

36 provinces fully missing, from ABRA to NCR, CITY OF MANILA, FIRST DISTRICT


province
NCR, FOURTH DISTRICT    0.131579
NCR, SECOND DISTRICT    0.530612
SULTAN KUDARAT          0.008130
Name: locality, dtype: float64

In [51]:
known = memberships.dropna(subset=["locality"])
missing = town_level[town_level["locality"].isna()]

has_history = missing["person_id"].isin(known["person_id"])
print("missing rows:", len(missing))
print("person has a known town in another year:", has_history.sum())

missing rows: 22022
person has a known town in another year: 13924


In [52]:
towns_per_person = known.groupby(["person_id", "province"])["locality"].nunique()

check = missing.merge(
    towns_per_person.rename("n_towns"),
    left_on=["person_id", "province"],
    right_index=True,
    how="left",
)
check["n_towns"].value_counts(dropna=False)

n_towns
1.0    13748
NaN     8194
2.0       79
3.0        1
Name: count, dtype: int64

### Summary: hf_memberships

**What it is:** One row per person, per position, per election. 86,234 rows, 8 columns (`id`, `person_id`, `party`, `region`, `province`, `locality`, `position`, `year`).

**Quality**
- Every `person_id` matches an `id` in `hf_persons` (0 orphans), so the two tables join cleanly.
- 5 elections, evenly covered: 2004 (17,372), 2007 (16,610), 2010 (17,507), 2013 (17,164), 2016 (17,581). No data after 2016.
- About 1,580 mayor rows per election, consistent with the roughly 1,600 Philippine cities and municipalities.

**Missing `locality` (27,640 rows, 32%)**
- **Expected blanks (5,618):** Provincial Board Member, House Representative, Governor, and Vice Governor cover a province or district, not one town.
- **Real gaps (22,022):** Councilor (17,678), Mayor (2,174), and Vice Mayor (2,170), each about 27% missing.
- **The gaps are concentrated in two years:**

| Year | Town-level rows missing locality |
|---|---|
| 2004, 2007, 2010 | 0% |
| 2013 | 15,842 of 16,043 (about 99%) |
| 2016 | 6,180 of 16,410 (about 38%) |

- In 2016, 36 provinces are fully missing, starting from ABRA and ending at "NCR, CITY OF MANILA, FIRST DISTRICT". This suggests the collection stopped partway through an alphabetical list (still to confirm whether every province in that range is affected). Partially missing: NCR Fourth District (13%), NCR Second District (53%), Sultan Kudarat (0.8%).

**Backfill potential from each person's own history** (same person, same province, town known in another year):

| Group | Rows | Share |
|---|---|---|
| Exactly one known town | 13,748 | 62.4% |
| Two or three known towns (ambiguous) | 80 | 0.4% |
| No known town in that province | 8,194 | 37.2% |

96 of the 8,194 have a known town in a *different* province (they moved, or the province name differs between years).

**Other observations**
- 39 House Representatives and 2 Provincial Board Members have a locality (to check).
- `position` values are written in ALL CAPS.
- NCR uses district-style province names, e.g. "NCR, CITY OF MANILA, FIRST DISTRICT".

## 3. openhalalan_winners

In [24]:
winners = pd.read_csv(RAW / "openhalalan_winners.csv")
print(winners.shape)
winners.info()

(157333, 14)
<class 'pandas.DataFrame'>
RangeIndex: 157333 entries, 0 to 157332
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   Last Name           157326 non-null  str  
 1   First Name          157251 non-null  str  
 2   Middle Name         139887 non-null  str  
 3   Middle Name Source  129693 non-null  str  
 4   Title               1140 non-null    str  
 5   Full Name           157333 non-null  str  
 6   Position            157333 non-null  str  
 7   Party               157203 non-null  str  
 8   Year                157333 non-null  int64
 9   Province            157281 non-null  str  
 10  City                143744 non-null  str  
 11  Region              157281 non-null  str  
 12  Sex                 148681 non-null  str  
 13  Sex Source          148681 non-null  str  
dtypes: int64(1), str(13)
memory usage: 31.4 MB


In [25]:
winners["Year"].value_counts().sort_index()

Year
2001    17490
2004    17368
2007    16608
2010    17502
2013    17161
2016    17774
2019    17793
2022    17818
2025    17819
Name: count, dtype: int64

In [26]:
pd.crosstab(winners["Year"], winners["City"].isna())

City,False,True
Year,,
2001,16394,1096
2004,15522,1846
2007,14912,1696
2010,15452,2050
2013,14905,2256
2016,16625,1149
2019,16630,1163
2022,16644,1174
2025,16660,1159


In [27]:
w2013 = winners[winners["Year"] == 2013]
w2013.loc[w2013["City"].isna(), "Position"].value_counts()

Position
COUNCILOR                           905
PROVINCIAL BOARD MEMBER             725
MEMBER, HOUSE OF REPRESENTATIVES    266
MAYOR                               124
VICE MAYOR                           85
VICE GOVERNOR                        78
GOVERNOR                             73
Name: count, dtype: int64

In [28]:
winners["Position"].value_counts()

Position
COUNCILOR                           118203
VICE MAYOR                           14419
MAYOR                                14398
PROVINCIAL BOARD MEMBER               6737
MEMBER, HOUSE OF REPRESENTATIVES      2108
VICE GOVERNOR                          714
GOVERNOR                               702
SENATOR                                 48
PRESIDENT                                2
VICE PRESIDENT                           2
Name: count, dtype: int64

In [29]:
no_province = winners[winners["Province"].isna()]
no_province["Position"].value_counts()

Position
SENATOR           48
PRESIDENT          2
VICE PRESIDENT     2
Name: count, dtype: int64

In [30]:
winners["Middle Name Source"].value_counts()

Middle Name Source
original       63033
comelec_lec    44986
dilg           13954
self-prior      7408
v8.5             312
Name: count, dtype: int64

In [31]:
hf_prov = set(memberships["province"].unique())
oh_prov = set(winners["Province"].dropna().unique())

print("HF provinces:", len(hf_prov), "| OpenHalalan provinces:", len(oh_prov))
print("Only in HF:", sorted(hf_prov - oh_prov))
print("Only in OpenHalalan:", sorted(oh_prov - hf_prov))

HF provinces: 85 | OpenHalalan provinces: 88
Only in HF: ['COMPOSTELA VALLEY', 'NCR, CITY OF MANILA, FIRST DISTRICT', 'NCR, FOURTH DISTRICT', 'NCR, SECOND DISTRICT', 'NCR, THIRD DISTRICT', 'TAWI-TAWI']
Only in OpenHalalan: ['DAVAO DE ORO', 'MAGUINDANAO DEL NORTE', 'MAGUINDANAO DEL SUR', 'NCR FIRST DISTRICT', 'NCR FOURTH DISTRICT', 'NCR SECOND DISTRICT', 'NCR THIRD DISTRICT', 'SPECIAL GEOGRAPHIC AREA', 'TAWI TAWI']


In [32]:
check_prov = ["DAVAO DE ORO", "MAGUINDANAO", "MAGUINDANAO DEL NORTE",
              "MAGUINDANAO DEL SUR", "SPECIAL GEOGRAPHIC AREA"]
winners[winners["Province"].isin(check_prov)].groupby("Province")["Year"].unique()

Province
DAVAO DE ORO               [2001, 2004, 2007, 2010, 2013, 2016, 2019, 202...
MAGUINDANAO                 [2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022]
MAGUINDANAO DEL NORTE                                                 [2025]
MAGUINDANAO DEL SUR                                                   [2025]
SPECIAL GEOGRAPHIC AREA                                               [2025]
Name: Year, dtype: object

In [33]:
hf_towns = set(memberships["locality"].dropna().unique())
oh_towns = set(winners["City"].dropna().unique())

only_hf = sorted(hf_towns - oh_towns)
only_oh = sorted(oh_towns - hf_towns)

print("HF towns:", len(hf_towns), "| OpenHalalan towns:", len(oh_towns))
print("Only in HF:", len(only_hf), "| Only in OpenHalalan:", len(only_oh))
print("Sample only in HF:", only_hf[:20])
print("Sample only in OpenHalalan:", only_oh[:20])

HF towns: 1463 | OpenHalalan towns: 1517
Only in HF: 254 | Only in OpenHalalan: 308
Sample only in HF: ['AL-BARKA', 'ALAMINOS CITY', 'ALFONSO LISTA (POTIA)', 'AMLAN (AYUQUITAN)', 'ANGELES CITY', 'ANINI-Y', 'ASUNCION (SAUG)', 'BACOLOD CITY', 'BACOLOD-KALAWI (BACOLOD GRANDE)', 'BACUNGAN (LEON T. POSTIGO)', 'BACUNGAN (Leon T. Postigo)', 'BAGO CITY', 'BAGUIO CITY', 'BAIS CITY', 'BALAGTAS (BIGAA)', 'BALANGA CITY', 'BALINDONG (WATU)', 'BANNA (ESPIRITU)', 'BASILISA (RIZAL)', 'BATAC CITY']
Sample only in OpenHalalan: ['AL BARKA', 'ALFONSO LISTA', 'ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN', 'AMLAN AYUQUITAN', 'ANGELES', 'ANINI Y', 'ANTIPOLO', 'ASUNCION', 'ASUNCION SAUG', 'BACOLOD KALAWI', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN', 'BACUNGAN LEON T POSTIGO', 'BAGO', 'BAGUIO', 'BAIS', 'BALAGTAS']


In [38]:
def norm_town(name):
    name = name.upper()
    name = re.sub(r"\(.*?\)", "", name)
    name = name.replace("Ñ", "N")                                  # new: Ñ → N
    name = name.replace("-", " ").replace(".", "").replace("'", "")  # new: also removes apostrophes
    name = re.sub(r"\s+", " ", name).strip()                       # moved: clean spaces BEFORE the CITY rules
    name = re.sub(r"^CITY OF ", "", name)
    name = re.sub(r" CITY$", "", name)
    return name

In [39]:
hf_norm = {norm_town(t) for t in hf_towns}
oh_norm = {norm_town(t) for t in oh_towns}

still_hf = sorted(hf_norm - oh_norm)
still_oh = sorted(oh_norm - hf_norm)

print("Before cleaning:", len(only_hf), "|", len(only_oh))
print("After cleaning:", len(still_hf), "|", len(still_oh))
print("Still only in HF:", still_hf[:20])
print("Still only in OpenHalalan:", still_oh[:20])

Before cleaning: 254 | 308
After cleaning: 5 | 120
Still only in HF: ['MAGADALENA', 'PATEROS TAGUIG', 'PRES CARLOS P GARCIA', 'SAM NARCISO', 'TAGUIG PATEROS']
Still only in OpenHalalan: ['ALFONSO LISTA POTIA', 'AMAI MANABILANG', 'AMAI MANABILANG BUMBARAN', 'AMLAN AYUQUITAN', 'ASUNCION SAUG', 'BACOLOD KALAWI BACOLOD GRANDE', 'BACUNGAN LEON T POSTIGO', 'BALAGTAS BIGAA', 'BALINDONG WATU', 'BALIWAG', 'BANAY BANAY', 'BASILISA RIZAL', 'BAYAWAN CITY TULONG', 'BINDOY PAYABON', 'BROOKE S POINT', 'BULALACAO SAN PEDRO', 'CALANASAN BAYAG', 'CORDOVA', 'D V CHIONGBIAN D M MARCOS', 'DARAGA LOCSIN']


In [40]:
winners["town_norm"] = winners["City"].map(norm_town, na_action="ignore")

overlap = winners[winners["Year"].between(2004, 2016)]
affected = overlap[overlap["town_norm"].isin(still_oh)]

print("Overlap-year rows with a leftover name:", len(affected), "of", len(overlap))
print("Distinct leftover names in overlap years:", affected["town_norm"].nunique())

Overlap-year rows with a leftover name: 3875 of 86413
Distinct leftover names in overlap years: 96


### Summary: openhalalan_winners

**What it is:** Election winners. 157,333 rows, 14 columns, covering 9 elections from 2001 to 2025 (about 17,000 rows each). Column names use Title Case with spaces (e.g. `Last Name`), unlike the snake_case names in HF.

**Strengths**
- **Town (`City`) is 91% complete**, including the years HF is missing: 2013 (14,905 of 17,161 rows, 87%) and 2016 (16,625 of 17,774 rows, 94%).
- **Has `Middle Name` (89% complete)**, which HF lacks. It is a useful kinship clue, since the middle name is usually the mother's maiden surname.

**Middle name sources**

| Source | Rows |
|---|---|
| original | 63,033 |
| comelec_lec | 44,986 |
| dilg | 13,954 |
| self-prior | 7,408 |
| v8.5 | 312 |

10,194 middle names have no recorded source. The meaning of `self-prior` and `v8.5` still needs to be confirmed from OpenHalalan's documentation.

**Gaps**
- In 2013, 1,114 town-level rows (905 Councilors, 124 Mayors, 85 Vice Mayors) have no town in this source either.
- 7 missing last names and 82 missing first names.
- The 52 rows with no Province/Region are national positions (48 Senators, 2 Presidents, 2 Vice Presidents). National coverage looks incomplete, so these rows are not useful for town-level analysis.
- `Sex`, `Sex Source`, and `Title` are not yet profiled.

**Matching with HF (2004–2016)**
- **No person ID**, so rows must be matched on name, province, year, and position.
- **Row counts are nearly identical** in shared years (within 5 rows, except 2016: 17,774 vs 17,581). The two sources likely share an origin, so their agreement is not independent confirmation.
- **Position names match HF exactly.** OpenHalalan also has SENATOR, PRESIDENT, VICE PRESIDENT.
- **Provinces:** 6 mappings needed: 4 NCR districts (comma and naming differences), TAWI-TAWI vs TAWI TAWI, and COMPOSTELA VALLEY (HF) vs DAVAO DE ORO (OpenHalalan, used for all years).
- **Towns:** mismatches dropped from 254 / 308 to 5 / 120 after the `norm_town` cleaning function (uppercase, remove parentheses, Ñ → N, remove hyphens, periods, and apostrophes, remove "CITY OF" / " CITY"). In shared years, 3,875 of 86,413 rows (4.5%) still use a leftover name across 96 distinct names. Together with HF's 5, about 100 manual mappings are needed.

**Open questions for the team**
- Use OpenHalalan's extra years (2001, 2019–2025), or limit the analysis to 2004–2016?
- How to handle provinces created after 2016: Maguindanao del Norte and del Sur (appear only in 2025; MAGUINDANAO runs through 2022) and Special Geographic Area (2025)?

## 4. psa_poverty (staging)